# 08 · Cache & Storage Levels em um Cluster Real

**Teoria**: docs/06-persistencia-e-otimizacao.md

**Pré-requisito**: `make spark` ainda em execução.

🎯 **Objetivo**: explorar os diferentes níveis de persistência do Spark (`MEMORY_ONLY` vs 
`MEMORY_AND_DISK`) e o impacto do **Adaptive Query Execution (AQE)** em dados distorcidos 
(data skew).

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("python-app")
    .remote("sc://localhost:15002")   # Endpoint gRPC do servidor Spark Connect
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
print("🖥️  Master UI:    http://localhost:8080")
print("🖥️  Worker UIs:   http://localhost:8081  http://localhost:8082")
print("📊 Spark App UI:  http://localhost:4040")

In [ ]:
# Lê cada tabela da camada Bronze em formato Parquet (colunar, comprimido)
# Criar Spark Data Frame
sdf_empresas = spark.read.parquet("/data/bronze/empresas")
sdf_funcionarios = spark.read.parquet("/data/bronze/funcionarios")
sdf_vendas = spark.read.parquet("/data/bronze/vendas")

## `MEMORY_ONLY` vs. `MEMORY_AND_DISK`

🧠 **Conceito**: `persist()` permite que você escolha exatamente onde os dados em cache residem. 
Verifique a aba **Storage** da Spark UI (http://localhost:4040/storage/) após cada célula para ver 
o DataFrame realmente materializado lá, com seu tamanho e fração de 
armazenamento.

| Nível | Descrição | Se não couber na RAM |
|---|---|---|
| `MEMORY_ONLY` | Apenas RAM | Recomputa do lineage (caro!) |
| `MEMORY_AND_DISK` | RAM + SSD | Derrama em disco (mais seguro) |

💡 **Dica**: `MEMORY_ONLY` é mais rápido se o dataset couber na RAM. 
`MEMORY_AND_DISK` é mais tolerante a picos de memória.

In [ ]:
from pyspark import StorageLevel

# Persiste em MEMORY_ONLY: dados ficam apenas na RAM dos executores
# Se não couber, partições excedentes são RECOMPUTADAS do lineage quando necessário
sdf_vendas.persist(StorageLevel.MEMORY_ONLY)
sdf_vendas.count()  # Materializa o cache — força a leitura e o armazenamento em memória
print("Cached with MEMORY_ONLY — check the Storage tab now.")

📌 **Verificação**:

Na aba **Storage** da Spark UI (http://localhost:4040/storage/), você deve ver 
o DataFrame `vendas` listado com:
- **Size in Memory**: tamanho ocupado na RAM
- **Size in Disk**: 0 B (pois é MEMORY_ONLY)
- **Fraction Cached**: fração do dataset que coube na RAM

Se `Fraction Cached < 1`, parte dos dados teria que ser recomputada se acessada novamente.

In [ ]:
# Limpa o cache anterior antes de aplicar novo StorageLevel
sdf_vendas.unpersist()

# Persiste em MEMORY_AND_DISK: RAM primeiro, SSD como fallback
# Se não couber na RAM, o Spark derrama o excedente em disco (mais seguro)
sdf_vendas.persist(StorageLevel.MEMORY_AND_DISK)
sdf_vendas.count()
print("Cached with MEMORY_AND_DISK — check the Storage tab again.")

📌 **Comparando os dois níveis**:

Na aba **Storage**, observe a diferença:
- Com `MEMORY_ONLY`: se `Fraction Cached < 1`, dados parciais podem ser perdidos
- Com `MEMORY_AND_DISK`: mesmo que não caiba tudo na RAM, o disco evita perda

💡 **Qual escolher?**
  - Dados que cabem na RAM → `MEMORY_ONLY` (mais rápido)
  - Dados grandes ou imprevisíveis → `MEMORY_AND_DISK` (mais seguro)
  - Cache entre stages de shuffle → `MEMORY_AND_DISK_SER` (serializado, mais compacto)

## Adaptive Query Execution (AQE) e data skew

🧠 **Problema**: dados distorcidos (skew) — uma partição muito maior que as outras — 
faz uma Task demorar muito mais, atrasando o Job inteiro (gargalo).

🎯 **Experimente**: construímos uma fatia deliberadamente distorcida — 90% das linhas caem 
em uma única região (`Sudeste`) — depois agregamos com AQE ligado e desligado.

📌 Observe a aba **Stages** da Spark UI. ⚠️ **Atenção**: a consulta abaixo é um `GROUP BY` 
puro, sem `join` — a otimização de AQE que entra em ação aqui é o **coalescing de partições 
pós-shuffle** (junta partições de saída pequenas/vazias para reduzir o número de Tasks 
agendadas), não o *Skew Join Optimization* (que só existe para joins e divide uma partição 
grande em sub-partições — você viu esse mecanismo específico no notebook 07, com 
`SortMergeJoin`). Aqui, a partição grande de `Sudeste` continua sendo processada por uma 
única Task mesmo com AQE ligado; o ganho vem de eliminar Tasks desperdiçadas nas partições 
quase vazias.

In [ ]:
import time
from pyspark.sql.functions import rand, when

# Libera o cache do exemplo anterior (MEMORY_AND_DISK) antes de seguir para o AQE
sdf_vendas.unpersist()

# Cria uma coluna com skew artificial: 90% de chance de cair em "Sudeste"
# vendas não tem coluna de região própria (isso vive em empresas) — geramos uma
# categoria sintética só para demonstrar o desbalanceamento
# seed=42 garante reprodutibilidade — mesmos números aleatórios toda execução
skewed = sdf_vendas.withColumn(
    "regiao_skewed",
    when(rand(seed=42) < 0.9, "Sudeste").otherwise("Outras"),
)
skewed.createOrReplaceTempView("vendas_skewed")

# Executa a mesma consulta com AQE ligado e desligado para comparar
for aqe in (False, True):
    # Alterna a configuração do AQE dinamicamente — sem precisar reiniciar a sessão
    spark.conf.set("spark.sql.adaptive.enabled", str(aqe).lower())
    start = time.perf_counter()
    spark.sql(
        "SELECT regiao_skewed, SUM(valor) FROM vendas_skewed GROUP BY regiao_skewed"
    ).collect()
    elapsed = time.perf_counter() - start
    print(f"AQE={aqe!s:5s} -> {elapsed:.2f}s (see Spark UI Stages tab for task-level detail)")

# Restaura o padrão — AQE ligado (recomendado para a maioria dos workloads)
spark.conf.set("spark.sql.adaptive.enabled", "true")

📌 **Análise do AQE**:

Com AQE **desligado**, o número de partições de shuffle fica fixo em 
`spark.sql.shuffle.partitions` (8, definido na SparkSession) — como só existem 2 valores 
distintos de `regiao_skewed`, 6 dessas 8 partições saem vazias, mas o Spark ainda agenda 
uma Task para cada uma.

Com AQE **ligado**, o Spark observa o tamanho real das partições após o shuffle e 
**coalesce** (funde) as pequenas/vazias, reduzindo o número de Tasks agendadas — é isso 
que explica boa parte do ganho de tempo aqui, não uma divisão da partição grande.

⚠️ **Não confunda com Skew Join Optimization:** essa é uma otimização *diferente* do AQE, 
específica para `join`s, que detecta um lado desproporcionalmente grande de um 
`SortMergeJoin`/`ShuffledHashJoin` e o **divide** em sub-partições menores processadas em 
paralelo. Como esta consulta não tem join nenhum, ela nunca entra em ação aqui — a Task que 
processa a fatia de `Sudeste` continua sendo uma só, com ou sem AQE.

💡 **Dica**: AQE está habilitado por padrão no Spark 3.4+. Desligá-lo pode fazer sentido 
apenas em cenários muito específicos (ex.: dados perfeitamente uniformes).

In [ ]:
# Encerra a sessão Spark Connect — libera recursos no cluster Docker
spark.stop()